In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import akshare as ak
from tqdm import tqdm
from datetime import datetime, timedelta
from collections import Counter
from quant_utils import midterm_entry_signal_backtrack,backtest_single_stock
from pathlib import Path
import tushare as ts

ts.set_token("103ef331836d4a8aa5d7485665c8f7503b8632c23748f1ba985c5713")
pro = ts.pro_api("103ef331836d4a8aa5d7485665c8f7503b8632c23748f1ba985c5713")



In [6]:
#单只股票的测试

code = "000001"

df = ak.stock_zh_a_hist(
    symbol=code,
    period="daily",
    adjust="qfq",
)
df['MA20'] = df['收盘'].rolling(20).mean()
df['MA60'] = df['收盘'].rolling(60).mean()

rets = backtest_single_stock(df)

print("信号次数:", len(rets))
print("平均收益:", np.mean(rets))
print("胜率:", np.mean(np.array(rets) > 0))

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [73]:
df 

,日期,股票代码,开盘,收盘,最高,最低,成交量,成交额,振幅,涨跌幅,涨跌额,换手率,MA20,MA60
0,1991-04-03,000001,-2.72,-2.72,-2.72,-2.72,1,5.000000e+03,0.00,2.86,0.08,0.00,NaN,NaN
1,1991-04-04,000001,-2.73,-2.73,-2.73,-2.73,3,1.500000e+04,0.00,-0.37,-0.01,0.00,NaN,NaN
2,1991-04-05,000001,-2.73,-2.73,-2.73,-2.73,2,1.000000e+04,0.00,0.00,0.00,0.00,NaN,NaN
3,1991-04-06,000001,-2.73,-2.73,-2.73,-2.73,7,3.400000e+04,0.00,0.00,0.00,0.00,NaN,NaN
4,1991-04-08,000001,-2.73,-2.73,-2.73,-2.73,2,1.000000e+04,0.00,0.00,0.00,0.00,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8334,2026-02-09,000001,11.05,11.07,11.11,11.01,619717,6.852182e+08,0.90,0.18,0.02,0.32,11.0575,11.400333
8335,2026-02-10,000001,11.07,11.06,11.10,11.02,600430,6.641402e+08,0.72,-0.09,-0.01,0.31,11.0370,11.388833
8336,2026-02-11,000001,11.06,11.07,11.09,11.02,431041,4.768019e+08,0.63,0.09,0.01,0.22,11.0225,11.378833
8337,2026-02-12,000001,11.07,10.96,11.08,10.93,681340,7.483385e+08,1.36,-0.99,-0.11,0.35,11.0050,11.368333


In [3]:
all_rets = []
stock_results = []

for _, row in tqdm(stock_list.iterrows(), total=len(stock_list)):
    code = row['code']
    
    try:
        df = ak.stock_zh_a_hist(
            symbol=code,
            period="daily",
            adjust="qfq",
        )
        
        if df.shape[0] < 100:
            continue
        
        df['MA20'] = df['收盘'].rolling(20).mean()
        df['MA60'] = df['收盘'].rolling(60).mean()

        rets = backtest_single_stock(df)

        if len(rets) == 0:
            continue

        all_rets.extend(rets)

        stock_results.append({
            "code": code,
            "信号次数": len(rets),
            "平均收益": np.mean(rets),
            "胜率": np.mean(np.array(rets) > 0)
        })

    except:
        continue


# ===== 整体统计 =====
print("总信号次数:", len(all_rets))
print("整体平均收益:", np.mean(all_rets))
print("整体胜率:", np.mean(np.array(all_rets) > 0))

100%|██████████| 100/100 [01:43<00:00,  1.03s/it]

总信号次数: 0
整体平均收益: nan
整体胜率: nan



/Users/ciciliu/anaconda3/envs/quant/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/ciciliu/anaconda3/envs/quant/lib/python3.10/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [13]:
stock_list = pd.read_csv("/Users/ciciliu/stock_analysis/data/mainboard_with_ts_code.csv",dtype={"code": "string"})
stock_list = stock_list.head(1000)
stock_list

,code,name,board,ts_code
0,000001,平安银行,深主板,000001.SZ
1,000002,万 科Ａ,深主板,000002.SZ
2,000004,*ST国华,深主板,000004.SZ
3,000006,深振业Ａ,深主板,000006.SZ
4,000007,全新好,深主板,000007.SZ
...,...,...,...,...
995,002500,山西证券,深主板,002500.SZ
996,002501,利源股份,深主板,002501.SZ
997,002506,协鑫集成,深主板,002506.SZ
998,002507,涪陵榨菜,深主板,002507.SZ


In [22]:
BY_STOCK_DIR = Path("/Users/ciciliu/stock_analysis/data/by_stock")

all_rets = []
stock_results = []
failed = []

# keep only needed columns from stock_list
cols = ["code"] + ["ts_code"] 
rows_df = stock_list[cols].copy()

# Last 25 years cutoff
cutoff_date = pd.Timestamp.today().normalize() - pd.DateOffset(years=25)

for row in tqdm(rows_df.itertuples(index=False), total=len(rows_df)):
    code = row.code
    ts_code = row.ts_code
    
    local_path = BY_STOCK_DIR / f"{ts_code}.parquet"
    if not local_path.exists():
        continue

    try:
        # read only required columns
        df = pd.read_parquet(local_path, columns=["trade_date", "close", "low", "vol"])

        # keep only last 25 years
        df["trade_date"] = pd.to_datetime(df["trade_date"], errors="coerce")
        df = df[df["trade_date"] >= cutoff_date]
        if len(df) < 100:
            continue

        df = df.sort_values("trade_date")
        df["MA20"] = df["close"].rolling(20).mean()
        df["MA60"] = df["close"].rolling(60).mean()

        rets = backtest_single_stock(df)  # backtest should use English cols now
        if not rets:
            continue

        all_rets.extend(rets)
        stock_results.append({
            "code": code,
            "ts_code": ts_code,
            "信号次数": len(rets),
            "平均收益": float(np.mean(rets)),
            "胜率": float(np.mean(np.array(rets) > 0)),
        })

    except Exception as e:
        failed.append((ts_code, str(e)))

print("failed:", len(failed))

100%|██████████| 1000/1000 [06:20<00:00,  2.63it/s]

failed: 0


In [ ]:
rows_df


,code,ts_code
0,000001,000001.SZ
1,000002,000002.SZ
2,000006,000006.SZ
3,000007,000007.SZ
4,000008,000008.SZ
...,...,...
995,002557,002557.SZ
996,002558,002558.SZ
997,002559,002559.SZ
998,002560,002560.SZ


In [23]:
#all_rets = [r for r in all_rets if abs(r) < 1]
wins = [r for r in all_rets if r > 0]
losses = [r for r in all_rets if r <= 0]

print("平均盈利:", np.mean(wins))
print("平均亏损:", np.mean(losses))
print("盈亏比:", abs(np.mean(wins) / np.mean(losses)))

平均盈利: 0.16531351283679405
平均亏损: -0.05282621805099478
盈亏比: 3.129383835072422


In [24]:
print("最大盈利:", np.max(all_rets))
print("最大亏损:", np.min(all_rets))

最大盈利: 3.7665580890336585
最大亏损: -0.6766743648960739


In [25]:
win_rate = len(wins) / len(all_rets) if len(rets) > 0 else np.nan
print("胜率:", win_rate)

胜率: 0.3275477239353891


In [6]:
stock_results

[{'code': '000001',
  'ts_code': '000001.SZ',
  '信号次数': 84,
  '平均收益': 0.01382230012252882,
  '胜率': 0.32142857142857145},
 {'code': '000002',
  'ts_code': '000002.SZ',
  '信号次数': 90,
  '平均收益': 0.031230936849909983,
  '胜率': 0.3888888888888889},
 {'code': '000006',
  'ts_code': '000006.SZ',
  '信号次数': 82,
  '平均收益': 0.03971712144887467,
  '胜率': 0.3780487804878049},
 {'code': '000007',
  'ts_code': '000007.SZ',
  '信号次数': 85,
  '平均收益': 0.028543192246946892,
  '胜率': 0.4470588235294118},
 {'code': '000008',
  'ts_code': '000008.SZ',
  '信号次数': 75,
  '平均收益': 0.16218402353175548,
  '胜率': 0.6533333333333333},
 {'code': '000009',
  'ts_code': '000009.SZ',
  '信号次数': 79,
  '平均收益': 0.08406417689514069,
  '胜率': 0.4050632911392405},
 {'code': '000010',
  'ts_code': '000010.SZ',
  '信号次数': 56,
  '平均收益': -0.03145060013621176,
  '胜率': 0.19642857142857142},
 {'code': '000011',
  'ts_code': '000011.SZ',
  '信号次数': 60,
  '平均收益': 0.004449420625480585,
  '胜率': 0.21666666666666667},
 {'code': '000012',
  'ts_code': 